In [1]:
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type
from eutils import EutilsNCBIError, EutilsRequestError

import pandas as pd 
import numpy as np 

from metapub import PubMedFetcher 

In [2]:
# Define NCBI error handling decorator with tenacity
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([EutilsNCBIError, EutilsRequestError])
)()

#Create function to retrieve ALL pmids (with NCBI error handling)
@retry_on_communication_error
def Get_list(query):
    fetch = PubMedFetcher()  
    num_of_articles = 500
    start_index = 0
    pmids = []
    while True:
        pmid_batch = fetch.pmids_for_query(query, 
                                      retstart=start_index,
                                      retmax=num_of_articles,
                                      pmc_only= False)
        pmids.extend(pmid_batch)
        start_index = len(pmids)
        if len(pmid_batch) < num_of_articles:
            break     
    return(pmids)

In [3]:
#Read-in query version
with open("PUBMED_query_v1.1", "r") as f:
    file = []
    for line in f:
        file.append(line.replace('\t','').replace('\n','').strip())
query = " ".join(file[1:])

In [4]:
#Run query

a = datetime.now()
START = "2000-01-01"
STOP = "2024-04-01"
start_date_str = START
pmid_list = []
while True: #define periods so that <10,000 are retrieved in the least request calls (best speed)
    if date.fromisoformat(start_date_str) <= date.fromisoformat("2002-07-01"): 
        month_interval = 6
    elif date.fromisoformat(start_date_str) <= date.fromisoformat("2005-11-01"):
        month_interval = 5
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2009-11-01")):
        month_interval = 4
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2011-10-01")):
        month_interval = 3
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2023-01-01")):
        month_interval = 2
    else:
        month_interval = 4
    next_start = date.fromisoformat(start_date_str) + relativedelta(months=month_interval)
    end_date = (next_start - relativedelta(days=1))
    end_date_str = end_date.strftime('%Y-%m-%d')
    date_str = f'''(("{start_date_str}"[Date - Publication] : "{end_date_str}"[Date - Publication]) '''
    pmids = Get_list(date_str+query)
    pmids_s = list(set(pmids))
    print(start_date_str,"-", end_date_str,": ", len(pmids_s), f"({len(pmids)})") #checks the number of PMIDs for each quarter(<10,000)
    pmid_list.extend(pmids_s)
    start_date_str = next_start.strftime('%Y-%m-%d')
    if next_start>=date.fromisoformat(STOP):
        end_date_str = STOP
        break
pmids = Get_list(query)
pmids_s = list(set(pmids))
pmid_list.extend(pmids_s)
print(start_date_str,"-", end_date_str,": ", len(pmids_s), f"({len(pmids)})") #checks the number of PMIDs for each quarter(<10,000)
print("Total query duration: ", datetime.now()-a)

2000-01-01 - 2000-06-30 :  7503 (7503)
2000-07-01 - 2000-12-31 :  7452 (7452)
2001-01-01 - 2001-06-30 :  8512 (8512)
2001-07-01 - 2001-12-31 :  8058 (8058)
2002-01-01 - 2002-06-30 :  8789 (8789)
2002-07-01 - 2002-12-31 :  8326 (8326)
2003-01-01 - 2003-05-31 :  8032 (8032)
2003-06-01 - 2003-10-31 :  7848 (7848)
2003-11-01 - 2004-03-31 :  8814 (8814)
2004-04-01 - 2004-08-31 :  8263 (8263)
2004-09-01 - 2005-01-31 :  9586 (9586)
2005-02-01 - 2005-06-30 :  8684 (8684)
2005-07-01 - 2005-11-30 :  9063 (9063)
2005-12-01 - 2006-03-31 :  8617 (8617)
2006-04-01 - 2006-07-31 :  7836 (7836)
2006-08-01 - 2006-11-30 :  8079 (8079)
2006-12-01 - 2007-03-31 :  9818 (9818)
2007-04-01 - 2007-07-31 :  8468 (8468)
2007-08-01 - 2007-11-30 :  8692 (8692)
2007-12-01 - 2008-03-31 :  9524 (9524)
2008-04-01 - 2008-07-31 :  8920 (8920)
2008-08-01 - 2008-11-30 :  8837 (8837)
2008-12-01 - 2009-03-31 :  9830 (9830)
2009-04-01 - 2009-07-31 :  8930 (8930)
2009-08-01 - 2009-11-30 :  8893 (8893)
2009-12-01 - 2010-02-28 :

In [6]:
#Issue

len(set(pmid_list))==len(pmid_list)

False

In [10]:
#Issue

#Confirm the same number is retrieved via Advanced Search at https://pubmed.ncbi.nlm.nih.gov/advanced/ 
len(set(pmid_list)) 
# ON THE SAME DAY March 11th, 20204, query up until now : 502,117
# NOT THE SAME

512116

In [ ]:
#save for permanent storage after issues are resolved:
date_tag = datetime.now().isoformat()[:10]
#np.savetxt('PMID_lists/pmids_'+ date_tag +.txt', list(set(pmid_list)), delimiter=",")